# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and systematically explore a Croissant dataset using the `mlcroissant` library, with references to all entities by their `@id` values for clarity and reproducibility.

### Dataset Source
The dataset Croissant schema is provided at:

[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and main records using the Croissant schema and the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset '{metadata.name}', Version: {metadata.version}")
print(metadata.description)

## 2. Data Overview
Let's review the available Record Sets and their fields, referencing each by its `@id`.

We will print each RecordSet's `@id`, `name`, and, if available, the associated Field `@id`s.

In [ ]:
# List all available record sets and their field '@id's.
print("Available Record Sets and Fields (referenced by @id):\n")
record_sets = []
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        print(f"- RecordSet @id: {rs['@id']}")
        name = rs.get('name', '(no name)')
        print(f"    name: {name}")
        fields = rs.get('fields', [])
        print(f"    field @ids: {[field['@id'] for field in fields] if fields else 'None'}")
        record_sets.append(rs['@id'])
else:
    # Try to infer record set ids from the dataset API
    # mlcroissant provides: dataset.record_sets()
    sets = list(dataset.record_sets())
    for rs in sets:
        print(f"- RecordSet @id: {rs['@id']}")
        name = rs.get('name', '(no name)')
        print(f"    name: {name}")
        fields = rs.get('fields', [])
        print(f"    field @ids: {[f['@id'] for f in fields] if fields else 'None'}")
        record_sets.append(rs['@id'])
if not record_sets:
    # Some croissant schemas have a main record set named by convention
    print("No explicit record sets found in metadata, trying to list data file objects...")
    record_sets = [rs['@id'] for rs in dataset.record_sets()] # might be empty or fail


## 3. Data Extraction
We can now extract Tabular Data from a specific RecordSet. For this dataset, the main record set typically aggregates protein tabular data. We will reference the main RecordSet by its `@id` as revealed above.

If you are unsure of the RecordSet `@id`, you may print all available record sets again before proceeding.

In [ ]:
#--- UPDATE THIS VARIABLE depending on what was listed above ---
# For this FAIR^2 SenScience dataset, the main data file typically has an @id like:
# 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'
# Use one of the record set @ids printed above. If none, use the main data file object @id.

# Example @id from distribution in metadata:
main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'

# For demonstration with mlcroissant, we add this id to a list:
record_sets = [main_record_set_id]  # Update this list with all discovered record set @ids if needed
dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")
        else:
            print(f"No records found for RecordSet @id: {record_set_id}")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

# Show column names of the main record set
if main_record_set_id in dataframes:
    print("Columns in main record set DataFrame:")
    print(dataframes[main_record_set_id].columns.tolist())
    # Show a preview of the table
    display(dataframes[main_record_set_id].head())


## 4. Exploratory Data Analysis (EDA)
Let's process and visualize some numeric fields referenced by their Croissant Field or Column `@id` values. We'll filter, normalize, and categorize as appropriate.

**Note:** The field and column names will match those printed above. Update the below variable with the actual numeric column name or field `@id` (e.g., `'mw'` for molecular weight).

We demonstrate:
 - Filtering records (e.g., proteins with molecular weight above a threshold)
 - Normalizing a numeric field
 - Optionally grouping by another categorical field


In [ ]:
#-- Choose a numeric field by its column name (@id) in the DataFrame --#
df = dataframes.get(main_record_set_id, pd.DataFrame())
print(f"Available columns: {df.columns.tolist()}")

# Typical mass spec tables have 'mw' (molecular weight) or similar columns
numeric_field_id = 'mw'  # Make sure this matches the actual column name; update if needed

# Filter records where molecular weight (mw) is above a threshold
threshold = 50000  # 50 kDa
if numeric_field_id in df.columns:
    filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
    print(f"Filtered to {len(filtered_df)} records with {numeric_field_id} > {threshold}")
    display(filtered_df.head())
    
    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
        ) / filtered_df[numeric_field_id].astype(float).std()
    print(f"Normalized '{numeric_field_id}' (z-score):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Optionally group by another field (e.g. 'description' or 'accession')
    group_field_id = 'description'  # Update as needed
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean '{numeric_field_id}' by '{group_field_id}':")
        display(grouped_df.head())
else:
    print(f"Field '{numeric_field_id}' not found in columns. Please update 'numeric_field_id' variable above.")

## 5. Visualization
We visualize the distribution of the chosen numeric field (e.g., molecular weight of proteins), and possibly its normalized counterpart.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    plt.figure(figsize=(10, 6))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=50, color='teal')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    if f'{numeric_field_id}_normalized' in filtered_df.columns:
        plt.figure(figsize=(10, 6))
        sns.histplot(filtered_df[f'{numeric_field_id}_normalized'].dropna(), bins=40, color='darkorange')
        plt.title(f'Normalized {numeric_field_id} (z-score)')
        plt.xlabel(f'{numeric_field_id}_normalized')
        plt.ylabel('Count')
        plt.show()


## 6. Conclusion
- We successfully loaded the SenScience FAIR^2 Mass Spectrometry dataset via Croissant schema using `mlcroissant`.
- All data elements were referenced by their Croissant `@id`, ensuring clarity for downstream work.
- Tabular protein data was extracted, filtered, normalized, and visualized to demonstrate exploratory data analysis typical for proteomics studies.
- For more advanced analyses, continue working with the DataFrames above and refer to Croissant `@id`s for reproducibility.
